In [1]:
pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

# Load individual cleaned tables from the provided path
base_path = Path(r"E:\my new for the me last for the change the my life for the life only goal\project\AI-Powered E-commerce Sales & Customer Analytics\databases\clean_data\data\clean")

customers = pd.read_csv(base_path / "customers.csv")
products = pd.read_csv(base_path / "products.csv")
orders = pd.read_csv(base_path / "orders.csv")
order_items = pd.read_csv(base_path / "order_items.csv")

print("Loaded tables:")
print(f"customers: {customers.shape}")
print(f"products: {products.shape}")
print(f"orders: {orders.shape}")
print(f"order_items: {order_items.shape}")

# Create engine and write all four tables into SQL
if 'engine' not in globals():
    engine = create_engine('sqlite:///ecommerce.db')  # change to your DB URI

for name, df in [
    ("customers", customers),
    ("products", products),
    ("orders", orders),
    ("order_items", order_items),
]:
    df.to_sql(name, con=engine, if_exists='replace', index=False)
    print(f"Wrote {name}: {len(df)} rows")

FileNotFoundError: [Errno 2] No such file or directory: 'E:\\my new for the me last for the change the my life for the life only goal\\project\\AI-Powered E-commerce Sales & Customer Analytics\\databases\\clean_data\\data\\clean\\customers.csv'

In [ ]:
pd.read_sql('SELECT * FROM customers LIMIT 5', con=engine)

,Customer ID,Country
0,13085.0,United Kingdom
1,13078.0,United Kingdom
2,15362.0,United Kingdom
3,18102.0,United Kingdom
4,12682.0,France


In [ ]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

base_path = Path(r"E:\my new for the me last for the change the my life for the life only goal\project\AI-Powered E-commerce Sales & Customer Analytics\databases\clean_data\data\clean")

table_files = {
    'customers': 'customers.csv',
    'products': 'products.csv',
    'orders': 'orders.csv',
    'order_items': 'order_items.csv',
}

def load_table(name, filename):
    if name in globals() and globals()[name] is not None:
        return globals()[name]
    path = base_path / filename
    if path.exists():
        return pd.read_csv(path)
    raise FileNotFoundError(f"Missing table '{name}' and CSV file not found at {path}")

if 'engine' not in globals():
    engine = create_engine('sqlite:///ecommerce.db')

for table_name, filename in table_files.items():
    df = load_table(table_name, filename)
    try:
        df.to_sql(table_name, engine, if_exists='replace', index=False)
        print(f"Wrote {table_name}: {len(df)} rows")
    except Exception as e:
        print(f"Failed writing {table_name}: {e}")

Wrote customers: 5878 rows
Wrote products: 4631 rows
Wrote orders: 36969 rows
Wrote order_items: 805549 rows


In [ ]:
pd.read_sql('SELECT * FROM customers LIMIT 5', con=engine)

,Customer ID,Country
0,13085.0,United Kingdom
1,13078.0,United Kingdom
2,15362.0,United Kingdom
3,18102.0,United Kingdom
4,12682.0,France


In [ ]:
pd.read_sql('SELECT * FROM orders LIMIT 5', con=engine)

,Invoice,Customer ID,InvoiceDate
0,489434,13085.0,2009-12-01 07:45:00
1,489435,13085.0,2009-12-01 07:46:00
2,489436,13078.0,2009-12-01 09:06:00
3,489437,15362.0,2009-12-01 09:08:00
4,489438,18102.0,2009-12-01 09:24:00


In [ ]:
pd.read_sql('SELECT * FROM order_items LIMIT 5', con=engine)

,Invoice,StockCode,Quantity,Price,Revenue
0,489434,85048,12,6.95,83.4
1,489434,79323P,12,6.75,81.0
2,489434,79323W,12,6.75,81.0
3,489434,22041,48,2.10,100.8
4,489434,21232,24,1.25,30.0


In [ ]:
pd.read_sql('SELECT * FROM products LIMIT 5', con=engine)

,StockCode,Description,Price
0,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,6.95
1,79323P,PINK CHERRY LIGHTS,6.75
2,79323W,WHITE CHERRY LIGHTS,6.75
3,22041,"RECORD FRAME 7"" SINGLE SIZE",2.10
4,21232,STRAWBERRY CERAMIC TRINKET BOX,1.25


In [ ]:
# Test query — total revenue by country
result = pd.read_sql(
    """
    SELECT c.Country AS country,
           COUNT(DISTINCT o.Invoice) AS total_orders,
           ROUND(SUM(oi.Revenue), 2) AS total_revenue
    FROM customers c
    JOIN orders o ON c."Customer ID" = o."Customer ID"
    JOIN order_items oi ON o.Invoice = oi.Invoice
    GROUP BY c.Country
    ORDER BY total_revenue DESC
    LIMIT 5
    """,
    con=engine,
)Z

print(result)

SyntaxError: invalid syntax (2922922390.py, line 15)

In [ ]:
for table in ['customers','products','orders','order_items']:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", engine)
    print(f"{table:15}: {count['n'][0]:>7,} rows")

customers      :   5,878 rows
products       :   4,631 rows
orders         :  36,969 rows
order_items    : 805,549 rows


In [ ]:
##  "SELECT queries are read-only operations and do not require transaction commits, whereas INSERT, UPDATE, and DELETE operations must be committed to persist changes."
## all about join table multiple tables together to get the desired result.

from sqlalchemy import text

with engine.connect() as conn:
    result = pd.read_sql(text("""
        SELECT c."Customer ID", c.Country,
               o.Invoice, o.InvoiceDate,
               p.Description,
               oi.Quantity, oi.Revenue
        FROM customers c
        JOIN orders o       ON c."Customer ID"  = o."Customer ID"
        JOIN order_items oi ON o.Invoice        = oi.Invoice
        JOIN products p     ON oi.StockCode     = p.StockCode
        LIMIT 5
    """), conn)

print(result)

   Customer ID         Country  Invoice          InvoiceDate  \
0      13085.0  United Kingdom   489434  2009-12-01 07:45:00   
1      13085.0  United Kingdom   489434  2009-12-01 07:45:00   
2      13085.0  United Kingdom   489434  2009-12-01 07:45:00   
3      13085.0  United Kingdom   489434  2009-12-01 07:45:00   
4      13085.0  United Kingdom   489434  2009-12-01 07:45:00   

                          Description  Quantity  Revenue  
0      STRAWBERRY CERAMIC TRINKET BOX        24     30.0  
1  FANCY FONT HOME SWEET HOME DOORMAT        10     59.5  
2                 SAVE THE PLANET MUG        24     30.0  
3        RECORD FRAME 7" SINGLE SIZE         48    100.8  
4          PINK DOUGHNUT TRINKET POT         24     39.6  


In [ ]:
# monthly revenue trend year month revenue
result = pd.read_sql("""
    SELECT
        strftime('%Y-%m', o.InvoiceDate) AS year_month,
        COUNT(DISTINCT o.Invoice)        AS total_orders,
        COUNT(DISTINCT o."Customer ID")  AS unique_customers,
        ROUND(SUM(oi.Revenue), 2)        AS total_revenue,
        ROUND(
            SUM(oi.Revenue) * 1.0 /
            NULLIF(COUNT(DISTINCT o.Invoice), 0), 2
        ) AS avg_order_value
    FROM orders o
    JOIN order_items oi 
        ON o.Invoice = oi.Invoice
    GROUP BY year_month
    ORDER BY year_month
""", engine)

In [ ]:
result

,year_month,total_orders,unique_customers,total_revenue,avg_order_value
0,2009-12,1512,955,686654.16,454.14
1,2010-01,1011,720,557319.06,551.26
2,2010-02,1104,772,506371.07,458.67
3,2010-03,1524,1057,699608.99,459.06
4,2010-04,1329,942,594609.19,447.41
5,2010-05,1377,966,599985.79,435.72
6,2010-06,1497,1041,639066.58,426.90
7,2010-07,1381,928,591636.74,428.41
8,2010-08,1293,911,604242.65,467.32
9,2010-09,1689,1145,831615.00,492.37


In [ ]:
# Reveals weekly purchasing pattern

import pandas as pd
from sqlalchemy import create_engine

result = pd.read_sql("""
    SELECT
        CASE strftime('%w', o.InvoiceDate)
            WHEN '0' THEN 'Sunday'    WHEN '1' THEN 'Monday'
            WHEN '2' THEN 'Tuesday'   WHEN '3' THEN 'Wednesday'
            WHEN '4' THEN 'Thursday'  WHEN '5' THEN 'Friday'
            WHEN '6' THEN 'Saturday'
        END                              AS day_of_week,
        COUNT(DISTINCT o.Invoice)        AS total_orders,
        ROUND(SUM(oi.Revenue), 2)        AS total_revenue,
        COUNT(DISTINCT o."Customer ID")  AS unique_customers
    FROM orders o
    JOIN order_items oi ON o.Invoice = oi.Invoice
    GROUP BY strftime('%w', o.InvoiceDate)
    ORDER BY total_revenue DESC
""", engine)

print(result)

  day_of_week  total_orders  total_revenue  unique_customers
0    Thursday          7773     3841082.96              3000
1     Tuesday          6627     3384678.83              2754
2   Wednesday          6649     3115619.69              2755
3      Monday          5755     2817407.81              2549
4      Friday          5387     2758423.49              2438
5      Sunday          4748     1816413.34              2128
6    Saturday            30        9803.05                26


In [ ]:
# Identifies top 10 products by revenue

import pandas as pd
from sqlalchemy import create_engine

# Step 1: Create engine
engine = create_engine("sqlite:///ecommerce.db")  # change to your DB path

# Step 2: Query
top_products = """
SELECT
    p.StockCode,
    p.Description,
    SUM(oi.Quantity)              AS total_units_sold,
    COUNT(DISTINCT oi.Invoice) AS times_ordered,
    ROUND(SUM(oi.Revenue), 2)     AS total_revenue
FROM order_items oi
JOIN products p 
    ON oi.StockCode = p.StockCode
GROUP BY p.StockCode, p.Description
ORDER BY total_revenue DESC
LIMIT 10
"""

# Step 3: Execute
result = pd.read_sql(top_products, engine)

print(result)

  StockCode                         Description  total_units_sold  \
0     22423            REGENCY CAKESTAND 3 TIER             24899   
1    85123A  WHITE HANGING HEART T-LIGHT HOLDER             93697   
2    85099B         JUMBO BAG RED WHITE SPOTTY              94983   
3     23843         PAPER CRAFT , LITTLE BIRDIE             80995   
4         M                              Manual              9803   
5     84879       ASSORTED COLOUR BIRD ORNAMENT             79913   
6      POST                             POSTAGE              5333   
7     47566                       PARTY BUNTING             23607   
8     23166      MEDIUM CERAMIC TOP STORAGE JAR             77916   
9     22086     PAPER CHAIN KIT 50'S CHRISTMAS              29477   

   times_ordered  total_revenue  
0           3317      286486.30  
1           4895      252227.81  
2           3260      170616.68  
3              1      168469.60  
4            620      152340.57  
5           2652      127074.17  
6 

In [ ]:
print(top_products)


SELECT
    p.StockCode,
    p.Description,
    SUM(oi.Quantity)              AS total_units_sold,
    COUNT(DISTINCT oi.Invoice) AS times_ordered,
    ROUND(SUM(oi.Revenue), 2)     AS total_revenue
FROM order_items oi
JOIN products p 
    ON oi.StockCode = p.StockCode
GROUP BY p.StockCode, p.Description
ORDER BY total_revenue DESC
LIMIT 10



In [ ]:
pd.read_sql("SELECT * FROM customers LIMIT 5", con=engine)

,Customer ID,Country
0,13085.0,United Kingdom
1,13078.0,United Kingdom
2,15362.0,United Kingdom
3,18102.0,United Kingdom
4,12682.0,France


In [ ]:
pd.read_sql("select * from orders limit 5", con=engine)

,Invoice,Customer ID,InvoiceDate
0,489434,13085.0,2009-12-01 07:45:00
1,489435,13085.0,2009-12-01 07:46:00
2,489436,13078.0,2009-12-01 09:06:00
3,489437,15362.0,2009-12-01 09:08:00
4,489438,18102.0,2009-12-01 09:24:00


In [ ]:
# Groups revenue by customer country.
result = pd.read_sql("""
SELECT 
    c."Country",
    COUNT(DISTINCT o."Invoice") AS total_orders,
    COUNT(DISTINCT c."Customer ID") AS unique_customers,
    ROUND(SUM(oi."Revenue"), 2) AS total_revenue,
    ROUND(
        SUM(oi."Revenue") / NULLIF(COUNT(DISTINCT c."Customer ID"), 0), 
        2
    ) AS revenue_per_customer
FROM customers c
JOIN orders o 
    ON c."Customer ID" = o."Customer ID"
JOIN order_items oi 
    ON o."Invoice" = oi."Invoice"
GROUP BY c."Country"
""", engine)

print(result)

                 Country  total_orders  unique_customers  total_revenue  \
0              Australia            83                14      164266.27   
1                Austria            47                12       26443.52   
2                Bahrain             4                 2        1354.37   
3                Belgium           160                27       69017.84   
4                 Brazil             2                 2        1411.87   
5                 Canada             8                 5        4883.04   
6        Channel Islands            55                13       44996.76   
7                 Cyprus            36                 9       22086.90   
8         Czech Republic             2                 1         826.74   
9                Denmark            56                11       75799.65   
10                  EIRE           569                 5      622354.96   
11    European Community             4                 1        1300.25   
12               Finland 

In [ ]:
# Order count and total spend per customer. Identifies loyal high-value customers vs one-time buyers.
result = pd.read_sql("""
SELECT
    c."Customer ID",
    c."Country",
    COUNT(DISTINCT o."Invoice") AS order_count,
    ROUND(SUM(oi."Revenue"), 2) AS total_spend,
    ROUND(AVG(oi."Revenue"), 2) AS avg_order_value,
    MIN(o."InvoiceDate") AS first_order,
    MAX(o."InvoiceDate") AS last_order
FROM customers c
JOIN orders o ON c."Customer ID" = o."Customer ID"
JOIN order_items oi ON o."Invoice" = oi."Invoice"
GROUP BY c."Customer ID", c."Country"
ORDER BY total_spend DESC
LIMIT 10
""", engine)

print(result)

   Customer ID         Country  order_count  total_spend  avg_order_value  \
0      18102.0  United Kingdom          145    608821.65           575.45   
1      14646.0     Netherlands          151    528602.52           137.34   
2      14156.0            EIRE          156    313946.37            77.56   
3      14911.0            EIRE          398    295972.63            26.32   
4      17450.0  United Kingdom           51    246973.09           582.48   
5      13694.0  United Kingdom          143    196482.81           128.84   
6      17511.0  United Kingdom           60    175603.55            91.89   
7      16446.0  United Kingdom            2    168472.50         56157.50   
8      16684.0  United Kingdom           55    147142.77           204.93   
9      12415.0       Australia           28    144458.37           156.00   

           first_order           last_order  
0  2009-12-01 09:24:00  2011-12-09 11:50:00  
1  2009-12-02 16:52:00  2011-12-08 12:12:00  
2  2009-12-01 

In [ ]:
result = pd.read_sql("""
SELECT
    c."Customer ID" AS customer_id,
    c.Country,
    COUNT(DISTINCT o.Invoice) AS order_count,
    ROUND(SUM(oi.Revenue), 2) AS total_spend,
    ROUND(
        SUM(oi.Revenue) / COUNT(DISTINCT o.Invoice), 
        2
    ) AS avg_order_value,
    MIN(o.InvoiceDate) AS first_order,   -- ✅ FIXED
    MAX(o.InvoiceDate) AS last_order     -- ✅ FIXED
FROM customers c
JOIN orders o 
    ON c."Customer ID" = o."Customer ID"
JOIN order_items oi 
    ON o.Invoice = oi.Invoice
GROUP BY c."Customer ID", c.Country
""", engine)

print(result)

      customer_id         Country  order_count  total_spend  avg_order_value  \
0         12346.0  United Kingdom           12     77556.46          6463.04   
1         12347.0         Iceland            8      5633.32           704.16   
2         12348.0         Finland            5      2019.40           403.88   
3         12349.0           Italy            4      4428.69          1107.17   
4         12350.0          Norway            1       334.40           334.40   
...           ...             ...          ...          ...              ...   
5873      18283.0  United Kingdom           22      2736.65           124.39   
5874      18284.0  United Kingdom            1       461.68           461.68   
5875      18285.0  United Kingdom            1       427.00           427.00   
5876      18286.0  United Kingdom            2      1296.43           648.22   
5877      18287.0  United Kingdom            7      4182.99           597.57   

              first_order           las

In [ ]:
# CHURN singnal: customers with no orders in the last 6 months
import pandas as pd

result = pd.read_sql("""
SELECT
    c."Customer ID",
    c.Country,
    MAX(o.InvoiceDate) AS last_order_date,
    CASE 
        WHEN MAX(o.InvoiceDate) < date('now','-6 months') THEN 'Churned'
        ELSE 'Active'
    END AS churn_status
FROM customers c
LEFT JOIN orders o 
    ON c."Customer ID" = o."Customer ID"
GROUP BY c."Customer ID", c.Country
""", engine)

In [ ]:
result


,Customer ID,Country,last_order_date,churn_status
0,12346.0,United Kingdom,2011-01-18 10:01:00,Churned
1,12347.0,Iceland,2011-12-07 15:52:00,Churned
2,12348.0,Finland,2011-09-25 13:13:00,Churned
3,12349.0,Italy,2011-11-21 09:51:00,Churned
4,12350.0,Norway,2011-02-02 16:01:00,Churned
...,...,...,...,...
5873,18283.0,United Kingdom,2011-12-06 12:02:00,Churned
5874,18284.0,United Kingdom,2010-10-04 11:33:00,Churned
5875,18285.0,United Kingdom,2010-02-17 10:24:00,Churned
5876,18286.0,United Kingdom,2010-08-20 11:57:00,Churned


In [ ]:
result = pd.read_sql("""
SELECT
    c."Customer ID" AS customer_id,
    c.Country,
    COUNT(DISTINCT o.invoice) AS order_count,
    ROUND(SUM(oi.revenue), 2) AS total_spend,
    ROUND(AVG(oi.revenue), 2) AS avg_order_value,
    ROUND(SUM(oi.quantity), 0) AS total_items,
    CAST(julianday(MAX(o.InvoiceDate)) -
         julianday(MIN(o.InvoiceDate))
    AS INTEGER) AS customer_tenure_days,
    CAST(julianday('2011-12-09') -
         julianday(MAX(o.InvoiceDate))
    AS INTEGER) AS days_inactive,
    CASE
        WHEN julianday('2011-12-09') -
             julianday(MAX(o.InvoiceDate)) > 90
        THEN 1 ELSE 0
    END AS churned
FROM customers c
JOIN orders o 
    ON c."Customer ID" = o."Customer ID"
JOIN order_items oi 
    ON o.invoice = oi.invoice
GROUP BY c."Customer ID", c.Country
ORDER BY churned DESC, days_inactive DESC
""", engine)

In [ ]:
result 

,customer_id,Country,order_count,total_spend,avg_order_value,total_items,customer_tenure_days,days_inactive,churned
0,17592.0,United Kingdom,1,148.30,11.41,38.0,0,737,1
1,17056.0,United Kingdom,1,128.60,16.07,26.0,0,737,1
2,14654.0,United Kingdom,1,246.86,9.14,273.0,0,737,1
3,13526.0,United Kingdom,2,1182.00,26.86,1911.0,0,737,1
4,12636.0,USA,1,141.00,141.00,1.0,0,737,1
...,...,...,...,...,...,...,...,...,...
5873,12518.0,Germany,5,2056.89,16.59,1318.0,109,0,0
5874,12437.0,France,39,12683.40,22.53,8000.0,735,0,0
5875,12433.0,Norway,10,20581.26,29.15,17116.0,476,0,0
5876,12423.0,Denmark,10,2622.39,15.89,1793.0,465,0,0


In [ ]:
result = pd.read_sql("""
SELECT
    churned,
    COUNT(*) AS customer_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS percentage
FROM (
    SELECT
        "Customer ID",
        CASE
            WHEN julianday('2011-12-09') -
                 julianday(MAX("InvoiceDate")) > 90
            THEN 1 ELSE 0
        END AS churned
    FROM orders
    GROUP BY "Customer ID"
)
GROUP BY churned
""", engine)

In [ ]:
result 

,churned,customer_count,percentage
0,0,2889,49.1
1,1,2989,50.9
